# Bronze → Silver → Gold — Reclamações ANATEL (Microsoft Fabric)

Mesmo dataset do [`telecom-powerbi-public`](../../README.md), agora processado como
arquitetura medallion num Lakehouse do Fabric, em PySpark.

**Pré-requisito**: os CSVs de `data/raw/` já foram enviados para `Files/raw/` deste
Lakehouse (veja `fabric-lakehouse-demo/README.md` para o passo a passo do portal).

| Camada | O que faz | Tabela Delta |
|---|---|---|
| Bronze | Ingestão bruta, sem transformação | `bronze_reclamacoes_scm`, `bronze_reclamacoes_smp` |
| Silver | União, limpeza, tipagem, normalização de operadora | `silver_reclamacoes` |
| Gold | KPIs agregados, prontos para BI | `gold_kpis_operadora`, `gold_kpis_regional`, `gold_kpis_temporal` |

## Bronze

Ingestão 1:1 do CSV de origem — sem limpar nada aqui de propósito. A camada Bronze existe
para poder reprocessar a Silver do zero sem voltar ao arquivo original, e para auditar o
dado exatamente como a ANATEL publicou.

In [ ]:
raw_files = {
    "scm": "Files/raw/reclamacoes_scm.csv",
    "smp": "Files/raw/reclamacoes_smp.csv",
}

for servico, caminho in raw_files.items():
    df_bronze = (
        spark.read
        .option("header", True)
        .option("sep", ";")
        .option("encoding", "ISO-8859-1")
        .csv(caminho)
    )
    (
        df_bronze.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(f"bronze_reclamacoes_{servico}")
    )
    print(f"bronze_reclamacoes_{servico}: {df_bronze.count()} linhas")

## Silver

Une os dois serviços (SCM + SMP), tipa a data, normaliza o nome da operadora para marca
(mesma regra de negócio do `src/kpis.py::_norm` do projeto Power BI, portada para Spark
para as duas versões não divergirem por acidente) e descarta linhas sem data válida.

In [ ]:
from pyspark.sql import functions as F

bronze_scm = spark.table("bronze_reclamacoes_scm").withColumn("servico", F.lit("SCM"))
bronze_smp = spark.table("bronze_reclamacoes_smp").withColumn("servico", F.lit("SMP"))
bronze = bronze_scm.unionByName(bronze_smp, allowMissingColumns=True)

# Mesma marca -> operadora do kpis.py (src/kpis.py::_norm), em Spark
nome_upper = F.upper(F.trim(F.col("Nome")))
operadora = (
    F.when(nome_upper.rlike("CLARO|NET SERVI|EMBRATEL"), "CLARO")
     .when(nome_upper.rlike("TELEFONICA|TELEF.NICA|VIVO"), "VIVO")
     .when(nome_upper.rlike("^TIM"), "TIM")
     .when(nome_upper.rlike("^OI "), "OI")
     .when(nome_upper.rlike("SERCOMTEL"), "SERCOMTEL")
     .otherwise("OUTROS")
)

silver = (
    bronze
    .withColumn("data_abertura", F.to_date("Data_Abertura", "dd/MM/yyyy"))
    .withColumn("operadora", operadora)
    .withColumn("motivo", F.initcap(F.trim(F.coalesce(F.col("Motivo"), F.lit("Outros")))))
    .withColumn("status", F.trim(F.coalesce(F.col("Status"), F.lit("Pendente"))))
    .withColumn("uf", F.upper(F.trim(F.coalesce(F.col("UF"), F.lit("XX")))))
    .withColumn("ano_mes", F.date_format("data_abertura", "yyyy-MM"))
    .filter(F.col("data_abertura").isNotNull())
    .select("data_abertura", "ano_mes", "servico", "operadora", "motivo", "status", "uf")
)

silver.write.format("delta").mode("overwrite").saveAsTable("silver_reclamacoes")
print(f"silver_reclamacoes: {silver.count()} linhas")

## Gold

KPIs agregados por operadora, por UF e por mês — o que o relatório de BI consome. Mesma
métrica de taxa de resolução usada no `.pbix` (`Status` contém "Respondida").

In [ ]:
silver = spark.table("silver_reclamacoes")
total = silver.count()

resolvida = F.when(F.col("status").contains("Respondida"), 1).otherwise(0)

gold_operadora = (
    silver
    .groupBy("operadora")
    .agg(
        F.count("*").alias("volume"),
        F.round(F.avg(resolvida) * 100, 2).alias("taxa_resolucao_pct"),
    )
    .withColumn("share_pct", F.round(F.col("volume") / F.lit(total) * 100, 2))
)
gold_operadora.write.format("delta").mode("overwrite").saveAsTable("gold_kpis_operadora")

gold_regional = (
    silver
    .groupBy("uf")
    .agg(
        F.count("*").alias("volume"),
        F.round(F.avg(resolvida) * 100, 2).alias("taxa_resolucao_pct"),
    )
    .withColumn("share_pct", F.round(F.col("volume") / F.lit(total) * 100, 2))
)
gold_regional.write.format("delta").mode("overwrite").saveAsTable("gold_kpis_regional")

gold_temporal = (
    silver
    .groupBy("ano_mes", "operadora")
    .agg(F.count("*").alias("volume"))
    .orderBy("ano_mes", "operadora")
)
gold_temporal.write.format("delta").mode("overwrite").saveAsTable("gold_kpis_temporal")

print(f"Total Silver: {total} linhas")
display(gold_operadora.orderBy(F.desc("volume")))

## Próximo passo

No workspace do Fabric: **Novo relatório → Power BI → escolher este Lakehouse como fonte
→ modo DirectLake**, apontando para as tabelas `gold_kpis_*`. O relatório consulta o
Delta Lake diretamente, sem importar/duplicar o dado — é a diferença de DirectLake para
o modo Import usado no `.pbix` principal deste repositório.

Conferir paridade: os números de `gold_kpis_operadora` devem bater com
`outputs/kpis_por_operadora.csv` (mesma fonte, mesmo período).